In [5]:
import gc
import json
import os
from tqdm import tqdm

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from transformers import AutoModel, AutoTokenizer
from transformers.models.roberta.modeling_roberta import RobertaModel
from transformers.models.roberta.tokenization_roberta import RobertaTokenizer
import torch

from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint, loguniform
from sklearn.metrics import mean_squared_error, r2_score

gc.collect()
torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Functions

In [ ]:
def evaluate_model(model, train_data, test_datasets, title, fit=True, figsize=(10, 10)):
    """
    Train and evaluate a regression model on multiple test datasets with scatter plots.
    
    Parameters
    ----------
    model : sklearn-like regressor
        The model to train and evaluate.
    train_data : tuple
        (X_train, y_train) data for fitting if fit=True.
    test_datasets : dict
        Dictionary of test datasets: {"name": (X_test, y_test), ...}.
    title : str
        Title for the entire figure.
    fit : bool, optional
        Whether to fit the model before evaluating (default: True).
    figsize : tuple, optional
        Figure size.
    """

    X_train, y_train = train_data
    if fit:
        model.fit(X_train, y_train)

    fig, axes = plt.subplots(2, 2, figsize=figsize, constrained_layout=True)
    axes = axes.flatten()

    for ax, (name, (X_test, y_test)) in zip(axes, test_datasets.items()):
        y_pred = model.predict(X_test)
        r2 = r2_score(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        pearson_corr = np.corrcoef(y_test, y_pred)[0, 1]

        ax.scatter(y_test, y_pred, alpha=0.6)
        ax.plot(y_test, y_test, color='red', linestyle='--')
        ax.set_xlim(0, 12)
        ax.set_ylim(0, 12)
        ax.set_title(f"{name.upper()}\nR²={r2:.3f}, RMSE={rmse:.3f}, PCC={pearson_corr:.3f}")
        ax.set_xlabel("True pK")
        ax.set_ylabel("Predicted pK")
        ax.grid(True, linestyle='--', alpha=0.5)

    fig.suptitle(title, fontsize=14, y=1.02)
    plt.show()

### **Generate Embeddings**

In [3]:
df = pd.read_csv(r"C:\ZHAW\PA2\Sequence_based_models\PDBbind_protein_pocket_ligands_pk.csv")
df

,pdb_id,protein_sequences,pocket_sequences,ligand_smiles,pK
0,10gs,['PYTVVYFPVRGRCAALRMLLADQGQSWKEEVVTVETWQEGSLKA...,"['VYFPVRGRCAALR', 'VVTVETWQEG', 'LKA', 'CLYGQL...",[NH3+][C@@H](CCC(=O)N[C@@H](CSCc1ccccc1)C(=O)N...,6.397940
1,11gs,['PYTVVYFPVRGRCAALRMLLADQGQSWKEEVVTVETWQEGSLKA...,"['VYFPVRGRCAALR', 'VVTVETWQEG', 'LKA', 'CLYGQL...",CC[C@@H](CSC[C@H](NC(=O)CC[C@H]([NH3+])C(=O)[O...,5.823909
2,13gs,['MPPYTVVYFPVRGRCAALRMLLADQGQSWKEEVVTVETWQEGSL...,"['YFPVRGRCAA', 'VVTVETWQE', 'LK', 'QLP', 'QSN'...",O=C([O-])C1=C/C(=N\Nc2ccc(S(=O)(=O)Nc3ccccn3)c...,4.619789
3,16pk,['EKKSINECDLKGKKVLIRVDFNVPVKNGKITNDYRIRSALPTLK...,"['GR', 'AFGTAHR', 'IVGGAKVSDKI', 'IGGAMAYT', '...",Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO[P@](=O)([O-])O[...,5.221849
4,184l,['MNIFEMLRIDEGLRLKIYKDTEGYYTIGIGHLLTKSPSLNAAKS...,"['AV', 'GILRNAKLKPVYDSLD', 'RRAAAINMVFQMGETGVA...",CC(C)Cc1ccccc1,4.721246
...,...,...,...,...,...
18576,966c,['RWEQTHLTYRIENYTPDLPRADVDHAIEKAFQLWSNVTPLTFTK...,"['LW', 'SPFDGPGGNLAHAFQ', 'DAHFD', 'EYNLHRVAAH...",O=C(CC1(S(=O)(=O)c2ccc(Oc3ccccc3)cc2)CCOCC1)NO,7.638272
18577,9abp,['NLKLGFLVKQPEEPWFQTEWKFADKAGKDLGFEVIKIAVPDGEK...,"['FLVKQPEEPWFQTE', 'ICTPDP', 'AVDDQ', 'LVMMAA'...",OC[C@H]1O[C@H](O)[C@H](O)[C@@H](O)[C@H]1O,8.000000
18578,9hvp,['PQITLWQRPLVTIKIGGQLKEALLDTGADDTVLEEMNLPGRWKP...,"['RPL', 'ALLDTGADDTVLE', 'KMIGGIGGFIKV', 'VL',...",CC(C)[C@H](N[C]([O-])OCc1ccccc1)C(=O)N[C@@H](C...,8.346787
18579,9icd,['SKVVVPAQGKKITLQNGKLNVPENPIIPYIEGDGIGVDVTPAML...,"['DGIGVDVT', 'AM', 'GPLTT', 'GGIGIAP', 'EATHGT...",Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)([O-])[O-])...,3.903090


#### **ESM-2** (Protein - Sequence Embeddings)

https://huggingface.co/facebook/esm2_t30_150M_UR50D?library=transformers

In [15]:
# ESM-2 Model and Tokenizer
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t30_150M_UR50D")
model = AutoModel.from_pretrained("facebook/esm2_t30_150M_UR50D").to(device)
model.eval()

# Generate ESM-2 embeddings for protein sequences
batch_size = 8
embeddings = []
for i in tqdm(range(0, len(df), batch_size), desc="Generating ESM-2 embeddings"):
    batch_seqs = df['protein_sequences'][i:i+batch_size].tolist()
    inputs = tokenizer(batch_seqs, return_tensors="pt", padding=True, truncation=True, max_length=1024)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        batch_emb = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
        embeddings.extend(batch_emb)

df['protein_embedding'] = embeddings

Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t30_150M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Generating ESM-2 embeddings: 100%|██████████| 2323/2323 [6:40:46<00:00, 10.35s/it]  


### **MolFormer - Embeddings**
- https://arxiv.org/abs/2405.04912
- https://huggingface.co/ibm-research/MoLFormer-XL-both-10pct

In [ ]:
model = AutoModel.from_pretrained("ibm/MoLFormer-XL-both-10pct", deterministic_eval=True, trust_remote_code=True)
tokenizer = AutoTokenizer.from_pretrained("ibm/MoLFormer-XL-both-10pct", trust_remote_code=True)

def get_molformer_embedding(smiles):
    inputs = tokenizer(smiles, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()} # Move inputs to device
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy() # Move back to CPU and convert to numpy (=> mean pooling)

# Generate embeddings
df['molformer_embedding'] = [
    get_molformer_embedding(smi) for smi in tqdm(df['ligand_smiles'], desc="Generating MoLFormer embeddings")
]